In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)


In [2]:
posts_df = pd.read_csv(OUTPUT_DIR / "posts_with_topics.csv")
print(posts_df.columns)


Index(['index', 'type', 'id', 'subreddit.id', 'subreddit.name',
       'subreddit.nsfw', 'created_utc', 'permalink', 'domain', 'url',
       'selftext', 'title', 'score', 'clean_text', 'topic'],
      dtype='object')


In [3]:
posts_df = posts_df.drop_duplicates(subset="id")

posts_df["created_utc"] = pd.to_datetime(
    posts_df["created_utc"], unit="s", errors="coerce"
)

In [4]:
posts_df["hour"] = posts_df["created_utc"].dt.hour
posts_df["dayofweek"] = posts_df["created_utc"].dt.dayofweek
posts_df["is_weekend"] = posts_df["dayofweek"].isin([5, 6]).astype(int)


In [5]:
# 텍스트 길이
posts_df["title_length"] = posts_df["title"].fillna("").str.len()
posts_df["selftext_length"] = posts_df["selftext"].fillna("").str.len()

posts_df["total_text_length"] = (
    posts_df["title_length"] + posts_df["selftext_length"]
)

In [6]:
# 토픽 피처
assert "topic" in posts_df.columns

topic_dummies = pd.get_dummies(posts_df["topic"], prefix="topic")
posts_df = pd.concat([posts_df, topic_dummies], axis=1)

In [7]:
posts_df["log_score"] = np.log1p(posts_df["score"])
posts_df["log_text_length"] = np.log1p(posts_df["total_text_length"])

In [8]:
final_cols = (
    [
        "id",
        "subreddit.name",
        "topic",
        "score",
        "log_score",
        "title_length",
        "selftext_length",
        "total_text_length",
        "log_text_length",
        "hour",
        "dayofweek",
        "is_weekend",
    ]
    + [c for c in posts_df.columns if c.startswith("topic_")]
)

posts_final = posts_df[final_cols]

In [9]:
posts_final.to_csv(
    OUTPUT_DIR / "posts_feature_engineered.csv",
    index=False
)

posts_final.head()

,id,subreddit.name,topic,score,log_score,title_length,selftext_length,total_text_length,log_text_length,hour,...,topic_156,topic_157,topic_158,topic_159,topic_160,topic_161,topic_162,topic_163,topic_164,topic_165
0,t4euxw,datasets,21,1,0.693147,89,522,611,6.416732,18,...,False,False,False,False,False,False,False,False,False,False
1,t47wiw,datasets,4,2,1.098612,79,123,202,5.313206,13,...,False,False,False,False,False,False,False,False,False,False
2,t3uct4,datasets,79,1,0.693147,65,544,609,6.413459,0,...,False,False,False,False,False,False,False,False,False,False
3,t3sxj4,datasets,78,1,0.693147,64,6,70,4.262680,23,...,False,False,False,False,False,False,False,False,False,False
4,t3mxt5,datasets,80,1,0.693147,45,916,961,6.869014,18,...,False,False,False,False,False,False,False,False,False,False
